In [ ]:
# ==============================================================================
# PIPELINE SIH/SUS + DUCKDB
# ==============================================================================

!pip install duckdb pandas pyarrow pysus numpy

In [ ]:
# ------------------------------------------------------------------------------
# CÉLULA 2 — Imports e configuração do ambiente
# ------------------------------------------------------------------------------

from pathlib import Path
import os
import sys
import json
import random
import shutil
import subprocess

# Instala automaticamente as dependências caso ainda não estejam disponíveis.
def instalar_dependencias():
    pacotes = {
        "duckdb": "duckdb",
        "pandas": "pandas",
        "pyarrow": "pyarrow",
        "numpy": "numpy",
        "pysus": "pysus",
    }

    faltantes = []

    for modulo, pacote in pacotes.items():
        try:
            __import__(modulo)
        except ImportError:
            faltantes.append(pacote)

    if faltantes:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *faltantes
        ])

instalar_dependencias()

import duckdb
import pandas as pd
import numpy as np

# O pysus expõe a função de download em módulos diferentes dependendo da
# versão instalada. Tentamos os caminhos mais comuns e caímos em erro claro
# caso nenhum funcione, em vez de um NameError silencioso mais adiante.
try:
    import pysus
    if not hasattr(pysus, "sih"):
        # Versões mais novas expõem o SIH em pysus.online_data.SIH
        from pysus.online_data import SIH as _pysus_sih_module

        class _PysusCompat:
            """Pequeno adaptador para manter a chamada pysus.sih(...)."""
            @staticmethod
            def sih(state, year, month, group="RD"):
                return _pysus_sih_module.download(
                    states=state, years=year, months=month, group=group
                )

        pysus.sih = _PysusCompat.sih
except ImportError as exc:
    raise ImportError(
        "Não foi possível importar o pacote 'pysus'. Confirme que "
        "'!pip install pysus' foi executado com sucesso antes deste ponto."
    ) from exc

# ------------------------------------------------------------------------------
# Diretórios do Google Colab
# ------------------------------------------------------------------------------

BASE_COLAB = Path("/content")

# Diretório onde devem estar os Parquets de entrada do SIH/SUS.
DIRETORIO_SIH = BASE_COLAB / "dados_sih"

# Diretório de saída do pipeline.
DIRETORIO_DADOS = BASE_COLAB / "dados"

# Datasets de treinamento.
DIRETORIO_DATASETS = BASE_COLAB / "datasets"

# Banco DuckDB persistente durante a sessão do Colab.
ARQUIVO_DUCKDB = DIRETORIO_DADOS / "pipeline_sih.duckdb"

DIRETORIO_SIH.mkdir(parents=True, exist_ok=True)
DIRETORIO_DADOS.mkdir(parents=True, exist_ok=True)
DIRETORIO_DATASETS.mkdir(parents=True, exist_ok=True)

_SIH_PATH = DIRETORIO_SIH.as_posix()

# ------------------------------------------------------------------------------
# Caminhos POSIX únicos e consistentes usados em todo o pipeline
# (evita misturar caminhos relativos soltos como 'dados/casos.parquet' com
# caminhos absolutos — tudo deriva das mesmas variáveis de diretório acima).
# ------------------------------------------------------------------------------
CAMINHO_CASOS = (DIRETORIO_DADOS / "casos.parquet").as_posix()
CAMINHO_PACIENTES_PARQUET = (DIRETORIO_DADOS / "pacientes_sinteticos.parquet").as_posix()
CAMINHO_PRONTUARIOS_PARQUET = (DIRETORIO_DADOS / "prontuarios_sinteticos.parquet").as_posix()
CAMINHO_PACIENTES_JSONL = (DIRETORIO_DADOS / "pacientes_sinteticos.jsonl").as_posix()
CAMINHO_PRONTUARIOS_JSONL = (DIRETORIO_DADOS / "prontuarios_sinteticos.jsonl").as_posix()
CAMINHO_RELATORIO_VALIDACAO = (DIRETORIO_DADOS / "relatorio_validacao_sintetico.csv").as_posix()

CAMINHO_TRAIN = (DIRETORIO_DATASETS / "train.jsonl").as_posix()
CAMINHO_VALIDATION = (DIRETORIO_DATASETS / "validation.jsonl").as_posix()
CAMINHO_TEST = (DIRETORIO_DATASETS / "test.jsonl").as_posix()
CAMINHO_DATASET_COMPLETO = (DIRETORIO_DATASETS / "dataset_treinamento.jsonl").as_posix()

# Mantidos por compatibilidade com nomes usados mais abaixo no arquivo.
_PATH_1 = CAMINHO_PACIENTES_PARQUET
_PATH_2 = CAMINHO_PRONTUARIOS_PARQUET


# ------------------------------------------------------------------------------
# Conexão ÚNICA (leitura e escrita) com DuckDB, usada apenas durante a
# construção do pipeline. Ela é fechada explicitamente ao final do script
# para liberar o arquivo — a partir daí, use a FerramentaConsultaSIH.
# ------------------------------------------------------------------------------

con = duckdb.connect(str(ARQUIVO_DUCKDB))

print("=" * 80)
print("AMBIENTE GOOGLE COLAB")
print("=" * 80)
print(f"Python: {sys.version.split()[0]}")
print(f"DuckDB: {duckdb.__version__}")
print(f"Diretório SIH: {DIRETORIO_SIH}")
print(f"Diretório de dados: {DIRETORIO_DADOS}")
print(f"Banco DuckDB: {ARQUIVO_DUCKDB}")
print()

In [ ]:
# ==============================================================================
# UTILITÁRIO — UPLOAD DE PARQUETS NO GOOGLE COLAB
# ==============================================================================

def upload_parquets_colab():
    """
    Abre o seletor de arquivos do Google Colab e copia os Parquets selecionados
    para /content/dados_sih/.
    """
    try:
        from google.colab import files
    except ImportError:
        print("Esta função só está disponível dentro do Google Colab.")
        return

    uploaded = files.upload()

    for nome, conteudo in uploaded.items():
        destino = DIRETORIO_SIH / nome
        destino.write_bytes(conteudo)
        print(f"Arquivo copiado para: {destino}")

    print(f"\nArquivos disponíveis em {DIRETORIO_SIH}:")
    for arquivo in sorted(DIRETORIO_SIH.glob("*.parquet")):
        print(f"  - {arquivo.name}")


def listar_arquivos_sih():
    arquivos = sorted(DIRETORIO_SIH.rglob("*.parquet"))

    print(f"Parquets encontrados: {len(arquivos)}")

    for arquivo in arquivos:
        print(f"  - {arquivo}")

    return arquivos

In [ ]:
# ==============================================================================
# 1. CONFIGURAÇÃO CENTRAL (altere apenas esta seção)
# ==============================================================================
ESTADO = "MG"
MUNICIPIO = "Teófilo Otoni"
ANOS = [2020, 2021, 2022]
MESES = list(range(1, 13))
GRUPO_SIH = "RD"

NUM_PACIENTES_SINTETICOS = 10000
SEED = 42

GERAR_JSONL = True
GERAR_PARQUET = True
GERAR_RELATORIO = True

DIVISAO_DATASET = {"train": 0.80, "validation": 0.10, "test": 0.10}

# Sementes para reprodutibilidade
random.seed(SEED)
np.random.seed(SEED)

print("=" * 80)
print("PIPELINE DE GERAÇÃO DE PRONTUÁRIOS SINTÉTICOS")
print("=" * 80)
print(f"[CONFIG] Estado: {ESTADO}, Município: {MUNICIPIO}, Anos: {ANOS}")
print(f"[CONFIG] Pacientes sintéticos a gerar: {NUM_PACIENTES_SINTETICOS}")
print(f"[CONFIG] Seed: {SEED}")
print("=" * 80)

In [ ]:
# ==============================================================================
# CAMADA 1 — DADOS DE REFERÊNCIA DO SIH/SUS
# ==============================================================================
print("\n" + "=" * 80)
print("CAMADA 1 — CARREGAMENTO DO SIH/SUS")
print("=" * 80)

def baixar_arquivos_sih(estado: str, anos: list[int], meses: list[int], grupo: str) -> list[str]:
    """Baixa (ou reaproveita do cache) os arquivos do SIH/SUS ano a ano."""
    todos_os_caminhos: list[str] = []
    for ano in anos:
        print(f"Baixando SIH/{grupo} - {estado} - {ano} (meses {meses[0]}-{meses[-1]})...")
        try:
            caminhos_ano = pysus.sih(state=estado, year=ano, month=meses, group=grupo)
        except TypeError:
            # Compatibilidade com versões do pysus sem parâmetro 'group'
            caminhos_ano = pysus.sih(state=estado, year=ano, month=meses)

        if isinstance(caminhos_ano, pd.DataFrame):
            raise RuntimeError(
                "pysus.sih() devolveu um DataFrame em vez de caminhos de arquivo. "
                "Verifique os parâmetros da instalação."
            )

        caminhos_ano = list(caminhos_ano)
        print(f" → {len(caminhos_ano)} arquivo(s) parquet disponível(is) para {ano}.")
        todos_os_caminhos.extend(caminhos_ano)

    if not todos_os_caminhos:
        raise RuntimeError("Nenhum arquivo foi retornado pelo pysus. Verifique conexão.")

    return todos_os_caminhos

caminhos_parquet = baixar_arquivos_sih(ESTADO, ANOS, MESES, GRUPO_SIH)
print(f"\n[1] Total de arquivos parquet (2020-2022): {len(caminhos_parquet)}")

print(f"Banco DuckDB conectado em: {ARQUIVO_DUCKDB}")

def _lista_sql(caminhos: list[str]) -> str:
    """Converte lista de caminhos em literal SQL do DuckDB."""
    escapados = [str(p).replace("'", "''") for p in caminhos]
    return "[" + ", ".join(f"'{p}'" for p in escapados) + "]"

_lista_caminhos_sql = _lista_sql(caminhos_parquet)
con.sql(f"""
CREATE OR REPLACE VIEW rd AS
SELECT * FROM read_parquet(
    {_lista_caminhos_sql},
    union_by_name = true,
    filename = true
)
""")

total_linhas = con.sql("SELECT COUNT(*) FROM rd").fetchone()[0]
print(f"Total de registros (AIHs) carregados na view 'rd': {total_linhas:,}")

# Verificação de colunas
colunas_disponiveis = {row[0] for row in con.sql("DESCRIBE rd").fetchall()}
print(f"\nTotal de colunas na base consolidada: {len(colunas_disponiveis)}")

colunas_essenciais = ["CNES", "MUNIC_MOV", "PROC_REA", "DIAG_PRINC"]
faltando = [c for c in colunas_essenciais if c not in colunas_disponiveis]
if faltando:
    raise RuntimeError(f"Colunas essenciais faltando: {faltando}")
print("Colunas essenciais confirmadas:", colunas_essenciais)

# Expressão para ano
if "ANO_CMPT" in colunas_disponiveis:
    EXPR_ANO = "CAST(ANO_CMPT AS INTEGER)"
elif "DT_INTER" in colunas_disponiveis:
    EXPR_ANO = "CAST(substr(CAST(DT_INTER AS VARCHAR), 1, 4) AS INTEGER)"
else:
    raise RuntimeError("Coluna de ano não encontrada (ANO_CMPT ou DT_INTER).")
print("Expressão usada para o ano do atendimento:", EXPR_ANO)

# Código do município
CODIGO_IBGE_7_DIGITOS = "3168606"
CODIGO_DATASUS_6_DIGITOS = CODIGO_IBGE_7_DIGITOS[:6]
print(f"\nCódigo IBGE (7 dígitos) de {MUNICIPIO}: {CODIGO_IBGE_7_DIGITOS}")
print(f"Código DATASUS (6 dígitos): {CODIGO_DATASUS_6_DIGITOS}")

validacao_municipio = con.sql(f"""
SELECT MUNIC_MOV, COUNT(*) AS quantidade
FROM rd
WHERE trim(CAST(MUNIC_MOV AS VARCHAR)) = '{CODIGO_DATASUS_6_DIGITOS}'
GROUP BY MUNIC_MOV
""").fetchall()

if not validacao_municipio:
    raise RuntimeError(
        f"Código '{CODIGO_DATASUS_6_DIGITOS}' não encontrado em MUNIC_MOV. "
        "Verifique a lista de municípios na base."
    )

codigo_municipio_confirmado, qtd_confirmada = validacao_municipio[0]
print(f"\nCódigo confirmado: '{codigo_municipio_confirmado}' ({qtd_confirmada:,} AIHs)")

# Regras de classificação por especialidade
def _regra_capitulo_cid(coluna: str, letra: str, ini: int, fim: int) -> str:
    """Gera condição SQL para faixa de CID-10."""
    return (
        f"({coluna} LIKE '{letra}%' "
        f"AND TRY_CAST(substr({coluna}, 2, 2) AS INTEGER) BETWEEN {ini} AND {fim})"
    )

COND_OBSTETRICIA = (
    f"({_regra_capitulo_cid('DIAG_PRINC', 'O', 0, 99)} "
    f"OR PROC_REA LIKE '0310%' OR PROC_REA LIKE '0411%')"
)
COND_GINECOLOGIA = _regra_capitulo_cid("DIAG_PRINC", "N", 70, 98)
COND_PEDIATRIA_PERINATAL = _regra_capitulo_cid("DIAG_PRINC", "P", 0, 96)
COND_PEDIATRIA_MALFORMACAO = _regra_capitulo_cid("DIAG_PRINC", "Q", 0, 99)

if "IDADE" in colunas_disponiveis and "COD_IDADE" in colunas_disponiveis:
    COND_PEDIATRIA_MALFORMACAO += (
        " AND (COD_IDADE <> 4 OR TRY_CAST(IDADE AS INTEGER) < 18)"
    )
COND_PEDIATRIA = f"({COND_PEDIATRIA_PERINATAL} OR {COND_PEDIATRIA_MALFORMACAO})"

CASE_ESPECIALIDADE = f"""
CASE
    WHEN {COND_OBSTETRICIA} THEN 'OBSTETRICIA'
    WHEN {COND_GINECOLOGIA} THEN 'GINECOLOGIA'
    WHEN {COND_PEDIATRIA} THEN 'PEDIATRIA'
    ELSE 'OUTROS'
END
"""

# Cria tabela de casos filtrada
con.sql(f"""
CREATE OR REPLACE TABLE casos AS
SELECT *,
    {EXPR_ANO} AS ano,
    {CASE_ESPECIALIDADE} AS especialidade
FROM rd
WHERE trim(CAST(MUNIC_MOV AS VARCHAR)) = '{codigo_municipio_confirmado}'
""")

con.sql("""
CREATE OR REPLACE TABLE casos AS
SELECT * FROM casos
WHERE especialidade IN ('OBSTETRICIA', 'GINECOLOGIA', 'PEDIATRIA')
""")

total_casos = con.sql("SELECT COUNT(*) FROM casos").fetchone()[0]
print(f"\n[2] Total de AIHs em {MUNICIPIO} classificadas nas 3 especialidades: {total_casos:,}")

# Exporta casos para referência
if GERAR_PARQUET:
    con.sql(f"COPY casos TO '{CAMINHO_CASOS}' (FORMAT PARQUET)")
    print(f"Tabela 'casos' exportada para: {CAMINHO_CASOS}")

In [ ]:
# ==============================================================================
# CAMADA 2 — ANÁLISE ESTATÍSTICA E GERAÇÃO DO PERFIL CLÍNICO SINTÉTICO
# ==============================================================================
print("\n" + "=" * 80)
print("CAMADA 2 — ANÁLISE ESTATÍSTICA E PERFIL CLÍNICO SINTÉTICO")
print("=" * 80)

# ------------------------------------------------------------------------------
# 2.1 Criação da tabela perfil_estatistico
# ------------------------------------------------------------------------------
print("\n[3] Calculando distribuições estatísticas...")

# Distribuição por especialidade
distrib_especialidade = con.sql("""
SELECT especialidade, COUNT(*) AS quantidade
FROM casos
GROUP BY especialidade
ORDER BY quantidade DESC
""").df()

distrib_especialidade["percentual"] = (
    distrib_especialidade["quantidade"] / distrib_especialidade["quantidade"].sum()
)
print("\nDistribuição por especialidade:")
print(distrib_especialidade.to_string(index=False))

# Distribuição por ano
distrib_ano = con.sql("""
SELECT ano, COUNT(*) AS quantidade
FROM casos
GROUP BY ano
ORDER BY ano
""").df()
distrib_ano["percentual"] = distrib_ano["quantidade"] / distrib_ano["quantidade"].sum()

# Distribuição por diagnóstico principal
distrib_diagnostico = con.sql("""
SELECT especialidade, DIAG_PRINC, COUNT(*) AS quantidade
FROM casos
WHERE DIAG_PRINC IS NOT NULL AND DIAG_PRINC != ''
GROUP BY especialidade, DIAG_PRINC
ORDER BY especialidade, quantidade DESC
""").df()

# Calcula percentual por especialidade
for esp in distrib_diagnostico["especialidade"].unique():
    mask = distrib_diagnostico["especialidade"] == esp
    total_esp = distrib_diagnostico.loc[mask, "quantidade"].sum()
    distrib_diagnostico.loc[mask, "percentual"] = (
        distrib_diagnostico.loc[mask, "quantidade"] / total_esp * 100
    )

# Distribuição por procedimento
distrib_procedimento = con.sql("""
SELECT especialidade, PROC_REA, COUNT(*) AS quantidade
FROM casos
WHERE PROC_REA IS NOT NULL AND PROC_REA != ''
GROUP BY especialidade, PROC_REA
ORDER BY especialidade, quantidade DESC
""").df()

for esp in distrib_procedimento["especialidade"].unique():
    mask = distrib_procedimento["especialidade"] == esp
    total_esp = distrib_procedimento.loc[mask, "quantidade"].sum()
    distrib_procedimento.loc[mask, "percentual"] = (
        distrib_procedimento.loc[mask, "quantidade"] / total_esp * 100
    )

# Distribuição por CNES
distrib_cnes = con.sql("""
SELECT CNES, COUNT(*) AS quantidade
FROM casos
WHERE CNES IS NOT NULL AND CNES != ''
GROUP BY CNES
ORDER BY quantidade DESC
""").df()
distrib_cnes["percentual"] = distrib_cnes["quantidade"] / distrib_cnes["quantidade"].sum()

# Distribuição por sexo (se disponível)
#
# O SIH/SUS registra SEXO como código numérico (1 = Masculino, 3 = Feminino,
# podendo variar conforme a fonte para M/F). A conversão para o rótulo por
# extenso é feita já nesta consulta SQL (em vez de só mais tarde, na geração
# dos pacientes sintéticos), para que qualquer relatório ou impressão desta
# tabela — inclusive esta mesma seção — já mostre 'Masculino'/'Feminino'.
if "SEXO" in colunas_disponiveis:
    distrib_sexo = con.sql("""
    SELECT
        especialidade,
        CASE
            WHEN trim(CAST(SEXO AS VARCHAR)) = '1' THEN 'Masculino'
            WHEN trim(CAST(SEXO AS VARCHAR)) = '3' THEN 'Feminino'
            WHEN upper(trim(CAST(SEXO AS VARCHAR))) IN ('M', 'MASCULINO') THEN 'Masculino'
            WHEN upper(trim(CAST(SEXO AS VARCHAR))) IN ('F', 'FEMININO') THEN 'Feminino'
            ELSE trim(CAST(SEXO AS VARCHAR))
        END AS SEXO,
        COUNT(*) AS quantidade
    FROM casos
    WHERE SEXO IS NOT NULL
    GROUP BY especialidade, SEXO
    ORDER BY especialidade, quantidade DESC
    """).df()
    for esp in distrib_sexo["especialidade"].unique():
        mask = distrib_sexo["especialidade"] == esp
        total_esp = distrib_sexo.loc[mask, "quantidade"].sum()
        distrib_sexo.loc[mask, "percentual"] = (
            distrib_sexo.loc[mask, "quantidade"] / total_esp * 100
        )
    print("\nDistribuição por sexo (por especialidade):")
    print(distrib_sexo.to_string(index=False))
else:
    distrib_sexo = None
    print("\nColuna SEXO não disponível — distribuição por sexo será estimada.")

# Distribuição por idade (se disponível)
if "IDADE" in colunas_disponiveis and "COD_IDADE" in colunas_disponiveis:
    distrib_idade = con.sql("""
    SELECT especialidade, IDADE, COD_IDADE, COUNT(*) AS quantidade
    FROM casos
    WHERE IDADE IS NOT NULL AND COD_IDADE IS NOT NULL
    GROUP BY especialidade, IDADE, COD_IDADE
    ORDER BY especialidade, quantidade DESC
    """).df()
    print("\nDistribuição por idade (amostra):")
    print(distrib_idade.head(20).to_string(index=False))
else:
    distrib_idade = None
    print("\nColunas IDADE/COD_IDADE não disponíveis — idade será estimada.")

# Distribuição de permanência (se disponível)
if "DIAPERM" in colunas_disponiveis:
    distrib_permanencia = con.sql("""
    SELECT especialidade, DIAPERM, COUNT(*) AS quantidade
    FROM casos
    WHERE DIAPERM IS NOT NULL AND DIAPERM >= 0
    GROUP BY especialidade, DIAPERM
    ORDER BY especialidade, DIAPERM
    """).df()
    for esp in distrib_permanencia["especialidade"].unique():
        mask = distrib_permanencia["especialidade"] == esp
        total_esp = distrib_permanencia.loc[mask, "quantidade"].sum()
        distrib_permanencia.loc[mask, "percentual"] = (
            distrib_permanencia.loc[mask, "quantidade"] / total_esp * 100
        )
    print("\nDistribuição de permanência (DIAPERM) — amostra:")
    print(distrib_permanencia.head(20).to_string(index=False))
else:
    distrib_permanencia = None

# Distribuição de desfecho (se disponível)
if "DT_SAIDA" in colunas_disponiveis or "MOTIVOSA" in colunas_disponiveis:
    if "MOTIVOSA" in colunas_disponiveis:
        distrib_desfecho = con.sql("""
        SELECT especialidade, MOTIVOSA, COUNT(*) AS quantidade
        FROM casos
        WHERE MOTIVOSA IS NOT NULL
        GROUP BY especialidade, MOTIVOSA
        ORDER BY especialidade, quantidade DESC
        """).df()
        for esp in distrib_desfecho["especialidade"].unique():
            mask = distrib_desfecho["especialidade"] == esp
            total_esp = distrib_desfecho.loc[mask, "quantidade"].sum()
            distrib_desfecho.loc[mask, "percentual"] = (
                distrib_desfecho.loc[mask, "quantidade"] / total_esp * 100
            )
        print("\nDistribuição de desfecho (MOTIVOSA):")
        print(distrib_desfecho.to_string(index=False))
    else:
        distrib_desfecho = None
else:
    distrib_desfecho = None

# Tabela diagnóstico → procedimento
diag_proc = con.sql("""
SELECT especialidade, DIAG_PRINC, PROC_REA, COUNT(*) AS quantidade
FROM casos
WHERE DIAG_PRINC IS NOT NULL AND DIAG_PRINC != ''
  AND PROC_REA IS NOT NULL AND PROC_REA != ''
GROUP BY especialidade, DIAG_PRINC, PROC_REA
ORDER BY especialidade, DIAG_PRINC, quantidade DESC
""").df()

# Calcula percentual por diagnóstico
for (esp, diag), group in diag_proc.groupby(["especialidade", "DIAG_PRINC"]):
    total_diag = group["quantidade"].sum()
    mask = (diag_proc["especialidade"] == esp) & (diag_proc["DIAG_PRINC"] == diag)
    diag_proc.loc[mask, "percentual"] = group["quantidade"] / total_diag * 100

# Salva perfil estatístico
perfil_estatistico = {
    "distrib_especialidade": distrib_especialidade,
    "distrib_ano": distrib_ano,
    "distrib_diagnostico": distrib_diagnostico,
    "distrib_procedimento": distrib_procedimento,
    "distrib_cnes": distrib_cnes,
    "distrib_sexo": distrib_sexo,
    "distrib_idade": distrib_idade,
    "distrib_permanencia": distrib_permanencia,
    "distrib_desfecho": distrib_desfecho,
    "diag_proc": diag_proc,
}

if GERAR_PARQUET:
    for nome, df in perfil_estatistico.items():
        if df is not None and isinstance(df, pd.DataFrame):
            caminho_perfil = (DIRETORIO_DADOS / f"perfil_{nome}.parquet").as_posix()
            df.to_parquet(caminho_perfil, index=False)
    print(f"\nPerfis estatísticos exportados para: {DIRETORIO_DADOS.as_posix()}/perfil_*.parquet")

print("\n[3] Distribuições estatísticas calculadas.")

In [ ]:
# ------------------------------------------------------------------------------
# 2.2 Funções de amostragem
# ------------------------------------------------------------------------------
def amostrar_especialidade(n: int, distrib: pd.DataFrame) -> list[str]:
    """Amostra especialidades respeitando a distribuição observada."""
    especialidades = distrib["especialidade"].tolist()
    pesos = distrib["percentual"].tolist()
    return list(np.random.choice(especialidades, size=n, p=pesos))

def amostrar_ano_por_especialidade(especialidade: str, distrib_ano: pd.DataFrame) -> int:
    """Amostra ano baseado na distribuição geral (pode ser refinado por especialidade)."""
    anos = distrib_ano["ano"].tolist()
    pesos = distrib_ano["percentual"].tolist()
    return int(np.random.choice(anos, p=pesos))

def amostrar_diagnostico(especialidade: str, distrib: pd.DataFrame) -> str:
    """Amostra diagnóstico baseado na distribuição por especialidade."""
    subset = distrib[distrib["especialidade"] == especialidade]
    if subset.empty:
        return "NÃO_ESPECIFICADO"
    diagnostico = subset["DIAG_PRINC"].tolist()
    pesos = subset["percentual"].tolist()
    # Normaliza
    total_p = sum(pesos)
    pesos = [p / total_p for p in pesos]
    return str(np.random.choice(diagnostico, p=pesos))

def amostrar_procedimento(especialidade: str, distrib: pd.DataFrame) -> str:
    """Amostra procedimento baseado na distribuição por especialidade."""
    subset = distrib[distrib["especialidade"] == especialidade]
    if subset.empty:
        return "NÃO_ESPECIFICADO"
    proc = subset["PROC_REA"].tolist()
    pesos = subset["percentual"].tolist()
    total_p = sum(pesos)
    pesos = [p / total_p for p in pesos]
    return str(np.random.choice(proc, p=pesos))

def amostrar_procedimento_por_diagnostico(
    especialidade: str, diagnostico: str, diag_proc: pd.DataFrame, fallback_distrib: pd.DataFrame
) -> str:
    """Amostra procedimento baseado no diagnóstico (com fallback para distribuição geral)."""
    subset = diag_proc[
        (diag_proc["especialidade"] == especialidade) &
        (diag_proc["DIAG_PRINC"] == diagnostico)
    ]
    if not subset.empty:
        proc = subset["PROC_REA"].tolist()
        pesos = subset["percentual"].tolist()
        total_p = sum(pesos)
        pesos = [p / total_p for p in pesos]
        return str(np.random.choice(proc, p=pesos))
    else:
        # Fallback
        return amostrar_procedimento(especialidade, fallback_distrib)

def amostrar_cnes(distrib: pd.DataFrame) -> str:
    """Amostra CNES baseado na distribuição observada."""
    cnes = distrib["CNES"].tolist()
    pesos = distrib["percentual"].tolist()
    return str(np.random.choice(cnes, p=pesos))

# Mapa para converter os códigos/abreviações de sexo (como vêm do SIH/SUS,
# ex.: 1 = Masculino, 3 = Feminino) para o rótulo por extenso.
MAPA_SEXO = {
    "1": "Masculino",
    "3": "Feminino",
    "M": "Masculino",
    "F": "Feminino",
    "MASCULINO": "Masculino",
    "FEMININO": "Feminino",
}

def normalizar_sexo(valor) -> str:
    """Converte um código ou abreviação de sexo para 'Masculino' ou 'Feminino'."""
    chave = str(valor).strip().upper()
    if chave in MAPA_SEXO:
        return MAPA_SEXO[chave]
    if chave.startswith("M"):
        return "Masculino"
    if chave.startswith("F"):
        return "Feminino"
    # Valor não reconhecido (ex.: ignorado/nulo): mantém o original como texto.
    return str(valor)

def amostrar_sexo(especialidade: str, distrib: pd.DataFrame) -> str:
    """Amostra sexo (por extenso: 'Masculino'/'Feminino') baseado na distribuição por especialidade."""
    if distrib is None:
        # Estimativa baseada em conhecimento clínico
        if especialidade == "OBSTETRICIA":
            return "Feminino"
        elif especialidade == "GINECOLOGIA":
            return "Feminino"
        elif especialidade == "PEDIATRIA":
            return np.random.choice(["Masculino", "Feminino"], p=[0.5, 0.5])

    subset = distrib[distrib["especialidade"] == especialidade]
    if subset.empty:
        return np.random.choice(["Masculino", "Feminino"], p=[0.5, 0.5])

    sexos = subset["SEXO"].tolist()
    pesos = subset["percentual"].tolist()
    total_p = sum(pesos)
    pesos = [p / total_p for p in pesos]
    sexo_amostrado = np.random.choice(sexos, p=pesos)
    return normalizar_sexo(sexo_amostrado)

def amostrar_idade(especialidade: str, distrib: pd.DataFrame) -> tuple[int, str]:
    """
    Amostra idade e retorna (idade_em_anos, faixa_etaria).
    Se distrib não estiver disponível, usa faixas clínicas típicas.
    """
    # Faixas etárias clínicas
    faixas_obstetrícia = [
        (12, 17, "12-17 anos"),
        (18, 24, "18-24 anos"),
        (25, 34, "25-34 anos"),
        (35, 44, "35-44 anos"),
    ]
    faixas_ginecologia = [
        (12, 17, "12-17 anos"),
        (18, 24, "18-24 anos"),
        (25, 34, "25-34 anos"),
        (35, 44, "35-44 anos"),
        (45, 59, "45-59 anos"),
        (60, 70, "60+ anos"),
    ]
    faixas_pediatria = [
        (0, 0, "0-28 dias"),  # recém-nascido
        (0, 1, "29 dias-1 ano"),
        (1, 4, "1-4 anos"),
        (5, 9, "5-9 anos"),
        (10, 14, "10-14 anos"),
        (15, 17, "15-17 anos"),
    ]

    if distrib is not None and not distrib.empty:
        subset = distrib[distrib["especialidade"] == especialidade]
        if not subset.empty:
            # Amostra idade a partir da distribuição
            idades = subset["IDADE"].tolist()
            cod_idades = subset["COD_IDADE"].tolist()
            pesos = subset["quantidade"].tolist()
            total_p = sum(pesos)
            pesos = [p / total_p for p in pesos]

            idx = np.random.choice(len(idades), p=pesos)
            idade_val = int(idades[idx])
            cod_idade_val = int(cod_idades[idx])

            # Converte para anos se necessário
            # COD_IDADE: 1=dias, 2=meses, 3=anos (pode variar — ajustar conforme documentação)
            if cod_idade_val == 1:  # dias
                idade_anos = 0
                faixa = "0-28 dias" if idade_val <= 28 else "29 dias-1 ano"
            elif cod_idade_val == 2:  # meses
                idade_anos = 0
                faixa = "29 dias-1 ano"
            else:  # anos
                idade_anos = idade_val
                # Determina faixa
                if especialidade == "PEDIATRIA":
                    for ini, fim, nome in faixas_pediatria:
                        if ini <= idade_anos <= fim:
                            faixa = nome
                            break
                    else:
                        faixa = "15-17 anos"
                elif especialidade == "OBSTETRICIA":
                    for ini, fim, nome in faixas_obstetrícia:
                        if ini <= idade_anos <= fim:
                            faixa = nome
                            break
                    else:
                        faixa = "25-34 anos"
                else:  # Ginecologia
                    for ini, fim, nome in faixas_ginecologia:
                        if ini <= idade_anos <= fim:
                            faixa = nome
                            break
                    else:
                        faixa = "25-34 anos"

            return idade_anos, faixa

    # Fallback: gera idade baseada em faixas típicas
    if especialidade == "OBSTETRICIA":
        faixa = random.choice([f[2] for f in faixas_obstetrícia])
        ini, fim, _ = [f for f in faixas_obstetrícia if f[2] == faixa][0]
        idade_anos = random.randint(ini, fim)
    elif especialidade == "PEDIATRIA":
        faixa = random.choice([f[2] for f in faixas_pediatria])
        if faixa == "0-28 dias":
            idade_anos = 0
        elif faixa == "29 dias-1 ano":
            idade_anos = 0
        else:
            ini, fim, _ = [f for f in faixas_pediatria if f[2] == faixa][0]
            idade_anos = random.randint(ini, fim)
    else:  # Ginecologia
        faixa = random.choice([f[2] for f in faixas_ginecologia])
        ini, fim, _ = [f for f in faixas_ginecologia if f[2] == faixa][0]
        idade_anos = random.randint(ini, fim)

    return idade_anos, faixa

def amostrar_permanencia(especialidade: str, distrib: pd.DataFrame) -> int:
    """Amostra tempo de permanência (DIAPERM)."""
    if distrib is not None and not distrib.empty:
        subset = distrib[distrib["especialidade"] == especialidade]
        if not subset.empty:
            diaperms = subset["DIAPERM"].tolist()
            pesos = subset["percentual"].tolist()
            total_p = sum(pesos)
            pesos = [p / total_p for p in pesos]
            return int(np.random.choice(diaperms, p=pesos))

    # Fallback: valores típicos
    if especialidade == "OBSTETRICIA":
        return random.choice([1, 2, 2, 3, 3, 3, 4, 5])
    elif especialidade == "PEDIATRIA":
        return random.choice([1, 2, 3, 3, 4, 5, 5, 7, 10])
    else:
        return random.choice([1, 1, 2, 2, 3, 3, 4])

def amostrar_desfecho(especialidade: str, distrib: pd.DataFrame) -> str:
    """Amostra desfecho (MOTIVOSA)."""
    if distrib is not None and not distrib.empty:
        subset = distrib[distrib["especialidade"] == especialidade]
        if not subset.empty:
            desfechos = subset["MOTIVOSA"].tolist()
            pesos = subset["percentual"].tolist()
            total_p = sum(pesos)
            pesos = [p / total_p for p in pesos]
            return str(np.random.choice(desfechos, p=pesos))

    # Fallback
    return random.choice(["Alta", "Alta", "Alta", "Transferência", "Óbito"])

In [ ]:
# ------------------------------------------------------------------------------
# 2.3 Geração dos pacientes sintéticos
# ------------------------------------------------------------------------------
print(f"\n[4] Gerando {NUM_PACIENTES_SINTETICOS} pacientes sintéticos...")

pacientes_sinteticos = []

for i in range(NUM_PACIENTES_SINTETICOS):
    paciente_id = f"PAC{i+1:08d}"

    # Amostra especialidade
    especialidade = amostrar_especialidade(1, distrib_especialidade)[0]

    # Amostra ano
    ano = amostrar_ano_por_especialidade(especialidade, distrib_ano)

    # Amostra diagnóstico
    diagnostico = amostrar_diagnostico(especialidade, distrib_diagnostico)

    # Amostra procedimento (baseado no diagnóstico)
    procedimento = amostrar_procedimento_por_diagnostico(
        especialidade, diagnostico, diag_proc, distrib_procedimento
    )

    # Amostra CNES
    cnes = amostrar_cnes(distrib_cnes)

    # Amostra sexo
    sexo = amostrar_sexo(especialidade, distrib_sexo)

    # Amostra idade e faixa etária
    idade, faixa_etaria = amostrar_idade(especialidade, distrib_idade)

    # Amostra permanência
    tempo_internacao = amostrar_permanencia(especialidade, distrib_permanencia)

    # Amostra desfecho
    desfecho = amostrar_desfecho(especialidade, distrib_desfecho)

    # Variáveis específicas por especialidade
    perfil_especifico = {}

    if especialidade == "OBSTETRICIA":
        # Variáveis obstétricas sintéticas
        idade_gestacional = random.randint(20, 42)  # semanas
        numero_gestacoes = random.randint(1, 5)
        numero_partos = random.randint(0, numero_gestacoes)
        numero_abortos = random.randint(0, max(0, numero_gestacoes - numero_partos))
        pre_natal = random.choice(["Sim", "Sim", "Sim", "Não"])  # 75% com pré-natal
        risco_gestacional = random.choice(["Baixo", "Baixo", "Médio", "Alto"])
        tipo_parto = random.choice(["Normal", "Normal", "Cesárea", "Cesárea"])

        perfil_especifico = {
            "idade_gestacional_semanas": idade_gestacional,
            "numero_gestacoes": numero_gestacoes,
            "numero_partos": numero_partos,
            "numero_abortos": numero_abortos,
            "pre_natal": pre_natal,
            "risco_gestacional": risco_gestacional,
            "tipo_parto": tipo_parto,
        }

    elif especialidade == "GINECOLOGIA":
        # Variáveis ginecológicas sintéticas (dependem do diagnóstico)
        ciclo_menstrual = random.choice(["Regular", "Irregular", "Ausente"])
        sangramento = random.choice(["Normal", "Aumentado", "Diminuído", "Ausente"])
        dor_pelvica = random.choice(["Não", "Leve", "Moderada", "Intensa"])
        corrimento = random.choice(["Não", "Sim"])
        sintomas_urinarios = random.choice(["Não", "Sim"])

        perfil_especifico = {
            "ciclo_menstrual": ciclo_menstrual,
            "sangramento": sangramento,
            "dor_pelvica": dor_pelvica,
            "corrimento": corrimento,
            "sintomas_urinarios": sintomas_urinarios,
        }

    elif especialidade == "PEDIATRIA":
        # Variáveis pediátricas (dependem da idade)
        if faixa_etaria in ["0-28 dias", "29 dias-1 ano"]:
            # Recém-nascido / lactente
            peso_kg = round(random.uniform(2.5, 10.0), 2)
            altura_cm = round(random.uniform(45, 75), 1)
            temperatura = round(random.uniform(36.0, 37.5), 1)
            frequencia_cardiaca = random.randint(100, 160)
            frequencia_respiratoria = random.randint(30, 60)
            saturacao = random.randint(95, 100)
        elif faixa_etaria in ["1-4 anos", "5-9 anos"]:
            # Criança
            peso_kg = round(random.uniform(10, 30), 2)
            altura_cm = round(random.uniform(75, 140), 1)
            temperatura = round(random.uniform(36.0, 37.5), 1)
            frequencia_cardiaca = random.randint(80, 140)
            frequencia_respiratoria = random.randint(20, 40)
            saturacao = random.randint(95, 100)
        else:
            # Adolescente
            peso_kg = round(random.uniform(30, 70), 2)
            altura_cm = round(random.uniform(140, 175), 1)
            temperatura = round(random.uniform(36.0, 37.5), 1)
            frequencia_cardiaca = random.randint(60, 100)
            frequencia_respiratoria = random.randint(16, 24)
            saturacao = random.randint(95, 100)

        perfil_especifico = {
            "peso_kg": peso_kg,
            "altura_cm": altura_cm,
            "temperatura": temperatura,
            "frequencia_cardiaca": frequencia_cardiaca,
            "frequencia_respiratoria": frequencia_respiratoria,
            "saturacao": saturacao,
        }

    paciente = {
        "id_paciente": paciente_id,
        "especialidade": especialidade,
        "ano": ano,
        "idade": idade,
        "faixa_etaria": faixa_etaria,
        "sexo": sexo,
        "diagnostico_principal": diagnostico,
        "procedimento_principal": procedimento,
        "cnes_referencia": cnes,
        "tempo_internacao": tempo_internacao,
        "desfecho": desfecho,
        **perfil_especifico,
    }

    pacientes_sinteticos.append(paciente)

In [ ]:
# ------------------------------------------------------------------------------
# 2.4 Persistência dos pacientes no DuckDB
# ------------------------------------------------------------------------------
# A lista Python é apenas a estrutura temporária de geração.
# O objeto oficial do pipeline passa a ser a tabela DuckDB.
df_pacientes_tmp = pd.DataFrame(pacientes_sinteticos)

# Registra o DataFrame temporário na mesma conexão DuckDB
con.register("pacientes_sinteticos_tmp", df_pacientes_tmp)

con.sql("""
CREATE OR REPLACE TABLE pacientes_sinteticos AS
SELECT *
FROM pacientes_sinteticos_tmp
""")

con.unregister("pacientes_sinteticos_tmp")
del df_pacientes_tmp

# Recupera o resultado através de SQL + .df()
df_pacientes = con.sql("""
SELECT *
FROM pacientes_sinteticos
ORDER BY id_paciente
""").df()

print(f"[5] {len(df_pacientes)} pacientes sintéticos gerados.")
print(f"[5] Tabela DuckDB 'pacientes_sinteticos' criada: {df_pacientes.shape}")

# Validação básica
print("\n[6] Aplicando regras de coerência...")

# Validação demográfica
assert (df_pacientes["idade"] >= 0).all(), "Idade negativa encontrada."
assert (df_pacientes["idade"] <= 120).all(), "Idade > 120 encontrada."

# Validação de especialidade
especialidades_validas = {"OBSTETRICIA", "GINECOLOGIA", "PEDIATRIA"}
assert df_pacientes["especialidade"].isin(especialidades_validas).all(), "Especialidade inválida."

# Validação obstétrica
mask_obst = df_pacientes["especialidade"] == "OBSTETRICIA"
if mask_obst.any():
    obst = df_pacientes[mask_obst]
    assert (obst["idade_gestacional_semanas"] >= 20).all(), "IG < 20 em obstetrícia."
    assert (obst["idade_gestacional_semanas"] <= 42).all(), "IG > 42 em obstetrícia."
    print("  ✓ Coerência obstétrica verificada.")

# Validação pediátrica
mask_ped = df_pacientes["especialidade"] == "PEDIATRIA"
if mask_ped.any():
    ped = df_pacientes[mask_ped]
    assert (ped["peso_kg"] > 0).all(), "Peso <= 0 em pediatria."
    assert (ped["temperatura"] >= 35).all(), "Temperatura < 35 em pediatria."
    assert (ped["temperatura"] <= 42).all(), "Temperatura > 42 em pediatria."
    print("  ✓ Coerência pediátrica verificada.")

print("[6] Regras de coerência aplicadas.")

# Exporta pacientes sintéticos diretamente pelo DuckDB
if GERAR_PARQUET:
    con.sql(f"""
    COPY pacientes_sinteticos
    TO '{CAMINHO_PACIENTES_PARQUET}'
    (FORMAT PARQUET)
    """)
    print(f"\nPacientes sintéticos exportados para: {CAMINHO_PACIENTES_PARQUET}")

In [ ]:
# ==============================================================================
# CAMADA 3 — GERAÇÃO DO PRONTUÁRIO TEXTUAL SINTÉTICO
# ==============================================================================
print("\n" + "=" * 80)
print("CAMADA 3 — GERAÇÃO DO PRONTUÁRIO TEXTUAL SINTÉTICO")
print("=" * 80)

# ------------------------------------------------------------------------------
# 3.1 Templates de sintomas por diagnóstico (mapeamento simplificado)
# ------------------------------------------------------------------------------
SINTOMAS_POR_DIAGNOSTICO = {
    # Obstetrícia
    "O80": ["Trabalho de parto espontâneo", "Contrações regulares", "Dilatação progressiva"],
    "O81": ["Trabalho de parto induzido", "Contrações regulares", "Ruptura de membranas"],
    "O82": ["Cesárea eletiva", "Sem trabalho de parto ativo", "Indicação médica"],
    "O10": ["Hipertensão arterial", "Edema", "Proteinúria"],
    "O14": ["Pré-eclâmpsia", "Hipertensão", "Proteinúria significativa"],
    "O24": ["Diabetes gestacional", "Glicemia elevada", "Rastreio positivo"],
    "O42": ["Ruptura prematura de membranas", "Perda de líquido amniótico"],
    "O45": ["Descolamento de placenta", "Sangramento vaginal", "Dor abdominal"],
    "O46": ["Placenta prévia", "Sangramento vaginal indolor"],
    "O60": ["Trabalho de parto prematuro", "Contrações antes de 37 semanas"],
    "O62": ["Distúrbio da contração uterina", "Trabalho de parto prolongado"],
    "O63": ["Trabalho de parto prolongado", "Estagnação da dilatação"],
    "O64": ["Desproporção cefalopélvica", "Trabalho de parto obstruído"],
    "O68": ["Comprometimento fetal", "Alteração da frequência cardíaca fetal"],
    "O70": ["Laceração perineal", "Trauma no parto"],
    "O71": ["Ruptura uterina", "Complicação grave do parto"],
    "O72": ["Hemorragia pós-parto", "Sangramento excessivo após parto"],
    "O75": ["Complicação do parto", "Condição materna adversa"],
    "O85": ["Sepse pós-parto", "Febre", "Infecção"],
    "O86": ["Infecção do trato genital pós-parto", "Febre", "Corrimento"],
    "O87": ["Complicação venosa pós-parto", "Trombose", "Edema"],
    "O88": ["Embolia obstétrica", "Complicação grave"],
    "O90": ["Complicação pós-parto", "Condição materna"],
    "O91": ["Infecção da mama", "Mastite", "Febre"],
    "O92": ["Problema de lactação", "Dificuldade de amamentação"],

    # Ginecologia
    "N70": ["Salpingite", "Dor pélvica", "Corrimento"],
    "N71": ["Doença inflamatória pélvica", "Dor abdominal", "Febre"],
    "N72": ["Cervicite", "Corrimento vaginal", "Sangramento"],
    "N73": ["Infecção pélvica", "Dor", "Febre"],
    "N74": ["Doença inflamatória pélvica por IST", "Dor", "Corrimento"],
    "N75": ["Abscesso da glândula de Bartholin", "Massa vulvar", "Dor"],
    "N76": ["Inflamação vaginal", "Corrimento", "Prurido"],
    "N77": ["Ulceracao vulvovaginal", "Lesão", "Dor"],
    "N80": ["Endometriose", "Dor pélvica", "Dismenorreia"],
    "N81": ["Prolapso genital", "Sensação de peso", "Incontinência"],
    "N82": ["Fístula genital", "Perda de líquido", "Infecção"],
    "N83": ["Cisto ovariano", "Dor pélvica", "Massa anexial"],
    "N84": ["Pólipo genital", "Sangramento irregular"],
    "N85": ["Distúrbio uterino", "Sangramento anormal"],
    "N86": ["Erosão cervical", "Sangramento de contato"],
    "N87": ["Displasia cervical", "Alteração no Papanicolau"],
    "N88": ["Distúrbio cervical", "Sangramento", "Dor"],
    "N89": ["Distúrbio vaginal", "Corrimento", "Prurido"],
    "N90": ["Distúrbio vulvar", "Lesão", "Prurido"],
    "N91": ["Amenorreia", "Ausência de menstruação"],
    "N92": ["Menstruação excessiva", "Sangramento abundante"],
    "N93": ["Sangramento uterino anormal", "Sangramento irregular"],
    "N94": ["Dismenorreia", "Dor menstrual", "Cólicas"],
    "N95": ["Distúrbio da menopausa", "Sintomas climatéricos"],
    "N96": ["Infertilidade", "Dificuldade de concepção"],
    "N97": ["Infertilidade feminina", "Tentativas sem sucesso"],
    "N98": ["Complicação de fertilização", "Procedimento de reprodução"],

    # Pediatria
    "P00": ["Afecção por condição placentária", "Comprometimento fetal"],
    "P01": ["Afecção por complicação do trabalho de parto"],
    "P02": ["Afecção por complicação da placenta"],
    "P03": ["Afecção por outras complicações do parto"],
    "P04": ["Afecção por agente nocivo"],
    "P05": ["Retardo de crescimento intrauterino", "Baixo peso"],
    "P07": ["Prematuridade", "Baixo peso ao nascer"],
    "P08": ["Macrossomia", "Peso elevado ao nascer"],
    "P09": ["Afecção por complicação da gestação"],
    "P10": ["Trauma intracraniano", "Hemorragia intracraniana"],
    "P11": ["Outro trauma do SNC", "Lesão neurológica"],
    "P12": ["Trauma do couro cabeludo", "Edema", "Hemorragia"],
    "P13": ["Trauma do esqueleto", "Fratura", "Lesão"],
    "P14": ["Trauma do sistema nervoso periférico"],
    "P15": ["Outro trauma obstétrico"],
    "P20": ["Hipóxia intrauterina", "Sofrimento fetal"],
    "P21": ["Asfixia ao nascer", "Depressão neonatal"],
    "P22": ["Síndrome do desconforto respiratório", "Dificuldade respiratória"],
    "P23": ["Pneumonia congênita", "Infecção respiratória"],
    "P24": ["Síndrome de aspiração neonatal"],
    "P25": ["Pneumotórax", "Ar no espaço pleural"],
    "P26": ["Hemorragia pulmonar"],
    "P27": ["Doença respiratória crônica"],
    "P28": ["Outra condição respiratória"],
    "P29": ["Distúrbio cardiovascular", "Arritmia", "Insuficiência"],
    "P35": ["Sepse congênita", "Infecção sistêmica"],
    "P36": ["Sepse bacteriana", "Infecção grave"],
    "P37": ["Outras infecções congênitas"],
    "P38": ["Onfalite", "Infecção do coto umbilical"],
    "P39": ["Outra infecção", "Febre", "Infecção"],
    "P50": ["Perda de sangue fetal", "Hemorragia"],
    "P51": ["Hemorragia umbilical"],
    "P52": ["Hemorragia intracraniana"],
    "P53": ["Doença hemorrágica"],
    "P54": ["Hemorragia digestiva"],
    "P55": ["Doença hemolítica"],
    "P56": ["Hidropsia fetal"],
    "P57": ["Kernicterus", "Icterícia grave"],
    "P58": ["Icterícia por hemólise"],
    "P59": ["Icterícia", "Hiperbilirrubinemia"],
    "P60": ["Coagulação intravascular disseminada"],
    "P61": ["Distúrbio hematológico", "Anemia", "Policitemia"],
    "P70": ["Hipoglicemia", "Glicemia baixa"],
    "P71": ["Distúrbio do cálcio/magnésio"],
    "P72": ["Distúrbio endócrino"],
    "P74": ["Distúrbio metabólico"],
    "P75": ["Íleo meconial"],
    "P76": ["Obstrução intestinal"],
    "P77": ["Enterocolite necrosante"],
    "P78": ["Distúrbio digestivo", "Vômitos", "Diarreia"],
    "P80": ["Hipotermia", "Temperatura baixa"],
    "P81": ["Distúrbio térmico", "Hipertermia"],
    "P83": ["Afecção da pele", "Lesão cutânea"],
    "P84": ["Problema de alimentação", "Dificuldade de sucção"],
    "P90": ["Convulsão neonatal", "Atividade convulsiva"],
    "P91": ["Distúrbio cerebral", "Letargia", "Irritabilidade"],
    "P92": ["Problema alimentar", "Intolerância"],
    "P94": ["Distúrbio do tônus muscular"],
    "P95": ["Morte fetal", "Óbito intrauterino"],
    "P96": ["Outra afecção neonatal"],

    "Q00": ["Anencefalia", "Malformação craniana grave"],
    "Q01": ["Encefalocele", "Herniação cerebral"],
    "Q02": ["Microcefalia", "Crânio pequeno"],
    "Q03": ["Hidrocefalia congênita", "Acúmulo de LCR"],
    "Q04": ["Malformação cerebral", "Anomalia estrutural"],
    "Q05": ["Espinha bífida", "Defeito do tubo neural"],
    "Q06": ["Malformação da medula"],
    "Q07": ["Malformação do SNC"],
    "Q10": ["Malformação ocular", "Anomalia do olho"],
    "Q11": ["Anoftalmia/Microftalmia"],
    "Q12": ["Catarata congênita"],
    "Q13": ["Malformação do segmento anterior"],
    "Q14": ["Malformação da retina"],
    "Q15": ["Outra malformação ocular"],
    "Q16": ["Malformação do ouvido"],
    "Q17": ["Outra malformação auricular"],
    "Q18": ["Malformação da face/crânio"],
    "Q20": ["Malformação cardíaca congênita"],
    "Q21": ["Defeito do septo cardíaco"],
    "Q22": ["Malformação das válvulas"],
    "Q23": ["Malformação arterial"],
    "Q24": ["Outra malformação cardíaca"],
    "Q25": ["Malformação de grandes artérias"],
    "Q26": ["Malformação venosa"],
    "Q27": ["Malformação vascular periférica"],
    "Q28": ["Malformação circulatória"],
    "Q30": ["Malformação nasal"],
    "Q31": ["Malformação laríngea"],
    "Q32": ["Malformação traqueobrônquica"],
    "Q33": ["Malformação pulmonar"],
    "Q34": ["Outra malformação respiratória"],
    "Q35": ["Fenda palatina"],
    "Q36": ["Fenda labial"],
    "Q37": ["Fenda labiopalatina"],
    "Q38": ["Malformação da língua/boca"],
    "Q39": ["Malformação do esôfago"],
    "Q40": ["Malformação do trato digestivo superior"],
    "Q41": ["Ausência/atresia intestinal"],
    "Q42": ["Ausência/atresia retal"],
    "Q43": ["Outra malformação intestinal"],
    "Q44": ["Malformação da vesícula/fígado"],
    "Q45": ["Malformação digestiva"],
    "Q50": ["Malformação dos órgãos genitais"],
    "Q51": ["Malformação uterina"],
    "Q52": ["Malformação genital feminina"],
    "Q53": ["Criptorquidia", "Testículo não descido"],
    "Q54": ["Hipospádia"],
    "Q55": ["Malformação genital masculina"],
    "Q56": ["Ambiguidade genital"],
    "Q60": ["Agenesia renal"],
    "Q61": ["Doença renal cística"],
    "Q62": ["Malformação obstrutiva renal"],
    "Q63": ["Outra malformação renal"],
    "Q64": ["Malformação urinária"],
    "Q65": ["Deformidade do pé"],
    "Q66": ["Deformidade do pé (outra)"],
    "Q67": ["Deformidade craniofacial"],
    "Q68": ["Deformidade osteomuscular"],
    "Q69": ["Polidactilia"],
    "Q70": ["Sindactilia"],
    "Q71": ["Defeito de redução do membro superior"],
    "Q72": ["Defeito de redução do membro inferior"],
    "Q73": ["Defeito de redução não especificado"],
    "Q74": ["Outra malformação dos membros"],
    "Q75": ["Malformação craniofacial"],
    "Q76": ["Malformação da coluna"],
    "Q77": ["Osteocondrodisplasia"],
    "Q78": ["Outra displasia óssea"],
    "Q79": ["Malformação musculoesquelética"],
    "Q80": ["Ictiose"],
    "Q81": ["Epidermólise bolhosa"],
    "Q82": ["Outra malformação da pele"],
    "Q83": ["Malformação da mama"],
    "Q84": ["Malformação ectodérmica"],
    "Q85": ["Síndrome genética"],
    "Q86": ["Síndrome congênita"],
    "Q87": ["Síndrome malformativa"],
    "Q89": ["Malformação congênita múltipla"],
    "Q90": ["Síndrome de Down", "Trissomia 21"],
    "Q91": ["Trissomia 18/13"],
    "Q92": ["Outra trissomia"],
    "Q93": ["Monossomia"],
    "Q95": ["Anomalia cromossômica"],
    "Q96": ["Síndrome de Turner"],
    "Q97": ["Outra anomalia cromossômica"],
    "Q98": ["Anomalia dos cromossomos sexuais"],
    "Q99": ["Outra anomalia cromossômica"],
}

def obter_sintomas(diagnostico: str, especialidade: str) -> list[str]:
    """Retorna lista de sintomas plausíveis para o diagnóstico."""
    # Tenta encontrar prefixo do diagnóstico
    prefixo = diagnostico[:3] if len(diagnostico) >= 3 else diagnostico

    if prefixo in SINTOMAS_POR_DIAGNOSTICO:
        return SINTOMAS_POR_DIAGNOSTICO[prefixo]

    # Fallback genérico por especialidade
    if especialidade == "OBSTETRICIA":
        return ["Condição obstétrica", "Acompanhamento", "Internação"]
    elif especialidade == "GINECOLOGIA":
        return ["Condição ginecológica", "Sintomas pélvicos", "Avaliação"]
    else:
        return ["Condição pediátrica", "Avaliação clínica", "Internação"]

In [ ]:
# ------------------------------------------------------------------------------
# 3.2 Geração do prontuário textual
# ------------------------------------------------------------------------------
print(f"\n[7] Gerando {NUM_PACIENTES_SINTETICOS} prontuários sintéticos...")

def gerar_prontuario(paciente: dict) -> str:
    """Gera prontuário textual estruturado a partir do perfil clínico."""

    id_paciente = paciente["id_paciente"]
    especialidade = paciente["especialidade"]
    idade = paciente["idade"]
    faixa_etaria = paciente["faixa_etaria"]
    sexo = paciente["sexo"]
    diagnostico = paciente["diagnostico_principal"]
    procedimento = paciente["procedimento_principal"]
    tempo_internacao = paciente["tempo_internacao"]
    desfecho = paciente["desfecho"]

    # Sintomas
    sintomas = obter_sintomas(diagnostico, especialidade)

    # Gera texto baseado na especialidade
    if especialidade == "OBSTETRICIA":
        idade_gestacional = paciente.get("idade_gestacional_semanas", 38)
        numero_gestacoes = paciente.get("numero_gestacoes", 1)
        numero_partos = paciente.get("numero_partos", 0)
        numero_abortos = paciente.get("numero_abortos", 0)
        pre_natal = paciente.get("pre_natal", "Sim")
        risco_gestacional = paciente.get("risco_gestacional", "Baixo")
        tipo_parto = paciente.get("tipo_parto", "Normal")

        prontuario = f"""PRONTUÁRIO CLÍNICO SINTÉTICO — Hospital de Referência de Teófilo Otoni

Identificação
----------------
Paciente: {id_paciente}
Idade: {idade} anos
Sexo: {sexo}
Especialidade: {especialidade}
Ano de referência: {paciente["ano"]}

Queixa Principal
----------------
{', '.join(sintomas[:2])}.

História da Doença Atual
----------------
Paciente gestante, {idade} anos, comparece para atendimento obstétrico.
Refere {', '.join(sintomas).lower()}.
Gestação de {idade_gestacional} semanas.

Antecedentes Obstétricos
----------------
Gestações: {numero_gestacoes}
Partos: {numero_partos}
Abortos: {numero_abortos}
Pré-natal: {pre_natal}
Risco gestacional: {risco_gestacional}

Exame Físico
----------------
Paciente em bom estado geral.
Altura uterina compatível com idade gestacional.
Batimentos cardíacos fetais presentes.

Exames Complementares
----------------
Ultrassonografia obstétrica realizada.
Exames laboratoriais de rotina.

Diagnóstico
----------------
{diagnostico} - {', '.join(sintomas[:2])}

Procedimentos
----------------
{procedimento} - {tipo_parto}

Conduta
----------------
Manter observação.
Monitorização fetal contínua.
Analgesia conforme necessidade.

Evolução
----------------
Paciente evoluiu bem no pós-operatório/pós-parto.
Sem intercorrências significativas.

Desfecho
----------------
{desfecho} após {tempo_internacao} dia(s) de internação.
"""

    elif especialidade == "GINECOLOGIA":
        ciclo_menstrual = paciente.get("ciclo_menstrual", "Regular")
        sangramento = paciente.get("sangramento", "Normal")
        dor_pelvica = paciente.get("dor_pelvica", "Não")
        corrimento = paciente.get("corrimento", "Não")
        sintomas_urinarios = paciente.get("sintomas_urinarios", "Não")

        prontuario = f"""PRONTUÁRIO CLÍNICO SINTÉTICO — Hospital de Referência de Teófilo Otoni

Identificação
----------------
Paciente: {id_paciente}
Idade: {idade} anos
Sexo: {sexo}
Especialidade: {especialidade}
Ano de referência: {paciente["ano"]}

Queixa Principal
----------------
{', '.join(sintomas[:2])}.

História da Doença Atual
----------------
Paciente comparece para avaliação ginecológica.
Refere {', '.join(sintomas).lower()}.
Ciclo menstrual: {ciclo_menstrual}.
Sangramento: {sangramento}.

Antecedentes Ginecológicos
----------------
Dor pélvica: {dor_pelvica}
Corrimento: {corrimento}
Sintomas urinários: {sintomas_urinarios}

Exame Físico
----------------
Paciente em bom estado geral.
Abdome flácido, indolor à palpação.
Exame especular realizado.

Exames Complementares
----------------
Ultrassonografia transvaginal.
Exames laboratoriais solicitados.

Diagnóstico
----------------
{diagnostico} - {', '.join(sintomas[:2])}

Procedimentos
----------------
{procedimento}

Conduta
----------------
Tratamento conforme protocolo.
Orientações sobre sinais de alarme.
Retorno agendado.

Evolução
----------------
Paciente evoluiu bem.
Sintomas em melhora.

Desfecho
----------------
{desfecho} após {tempo_internacao} dia(s) de internação.
"""

    else:  # PEDIATRIA
        peso_kg = paciente.get("peso_kg", 10)
        altura_cm = paciente.get("altura_cm", 80)
        temperatura = paciente.get("temperatura", 37.0)
        frequencia_cardiaca = paciente.get("frequencia_cardiaca", 120)
        frequencia_respiratoria = paciente.get("frequencia_respiratoria", 30)
        saturacao = paciente.get("saturacao", 98)

        prontuario = f"""PRONTUÁRIO CLÍNICO SINTÉTICO — Hospital de Referência de Teófilo Otoni

Identificação
----------------
Paciente: {id_paciente}
Idade: {faixa_etaria}
Sexo: {sexo}
Especialidade: {especialidade}
Ano de referência: {paciente["ano"]}

Queixa Principal
----------------
{', '.join(sintomas[:2])}.

História da Doença Atual
----------------
Paciente pediátrico comparece para avaliação.
Refere {', '.join(sintomas).lower()}.
Evolução conforme quadro clínico.

Antecedentes
----------------
Sem alergias conhecidas.
Vacinação em dia.

Exame Físico
----------------
Peso: {peso_kg} kg
Altura: {altura_cm} cm
Temperatura: {temperatura}°C
FC: {frequencia_cardiaca} bpm
FR: {frequencia_respiratoria} irpm
SatO2: {saturacao}%

Exames Complementares
----------------
Exames laboratoriais solicitados.
Imagem conforme necessidade.

Diagnóstico
----------------
{diagnostico} - {', '.join(sintomas[:2])}

Procedimentos
----------------
{procedimento}

Conduta
----------------
Tratamento de suporte.
Monitorização de sinais vitais.
Hidratação conforme necessidade.

Evolução
----------------
Paciente evoluiu bem.
Sinais vitais estáveis.

Desfecho
----------------
{desfecho} após {tempo_internacao} dia(s) de internação.
"""

    return prontuario

# Gera todos os prontuários
prontuarios_sinteticos = []
for paciente in pacientes_sinteticos:
    id_prontuario = paciente["id_paciente"].replace("PAC", "PRONT")
    prontuario_texto = gerar_prontuario(paciente)

    prontuarios_sinteticos.append({
        "id_prontuario": id_prontuario,
        "id_paciente": paciente["id_paciente"],
        "especialidade": paciente["especialidade"],
        "prontuario_texto": prontuario_texto,
    })

In [ ]:
# ------------------------------------------------------------------------------
# 3.3 Persistência dos prontuários no DuckDB
# ------------------------------------------------------------------------------
df_prontuarios_tmp = pd.DataFrame(prontuarios_sinteticos)

con.register("prontuarios_sinteticos_tmp", df_prontuarios_tmp)

con.sql("""
CREATE OR REPLACE TABLE prontuarios_sinteticos AS
SELECT *
FROM prontuarios_sinteticos_tmp
""")

con.unregister("prontuarios_sinteticos_tmp")
del df_prontuarios_tmp

# DataFrame usado pelo restante do pipeline é obtido do DuckDB
df_prontuarios = con.sql("""
SELECT *
FROM prontuarios_sinteticos
ORDER BY id_paciente
""").df()

print(f"[7] {len(df_prontuarios)} prontuários sintéticos gerados.")
print(f"[7] Tabela DuckDB 'prontuarios_sinteticos' criada: {df_prontuarios.shape}")

# Exporta prontuários diretamente pelo DuckDB
if GERAR_PARQUET:
    con.sql(f"""
    COPY prontuarios_sinteticos
    TO '{CAMINHO_PRONTUARIOS_PARQUET}'
    (FORMAT PARQUET)
    """)
    print(f"\nProntuários exportados para: {CAMINHO_PRONTUARIOS_PARQUET}")

if GERAR_JSONL:
    # Pacientes em JSONL
    with open(CAMINHO_PACIENTES_JSONL, "w", encoding="utf-8") as f:
        for paciente in pacientes_sinteticos:
            f.write(json.dumps(paciente, ensure_ascii=False) + "\n")

    # Prontuários em JSONL
    with open(CAMINHO_PRONTUARIOS_JSONL, "w", encoding="utf-8") as f:
        for pront in prontuarios_sinteticos:
            f.write(json.dumps(pront, ensure_ascii=False) + "\n")

    print(f"Arquivos JSONL exportados para: {CAMINHO_PACIENTES_JSONL} e {CAMINHO_PRONTUARIOS_JSONL}")

In [ ]:
# ==============================================================================
# GERAÇÃO DO DATASET PARA TREINAMENTO DE IA
# ==============================================================================
print("\n" + "=" * 80)
print("DATASET PARA TREINAMENTO DE IA")
print("=" * 80)

print("\n[8] Criando dataset de treinamento...")

def criar_tarefa_classificacao(prontuario: str, especialidade: str) -> dict:
    return {
        "instruction": "Identifique a especialidade do atendimento.",
        "input": prontuario,
        "output": especialidade,
    }

def criar_tarefa_extracao_diagnostico(prontuario: str, diagnostico: str) -> dict:
    return {
        "instruction": "Extraia o diagnóstico principal do prontuário.",
        "input": prontuario,
        "output": diagnostico,
    }

def criar_tarefa_extracao_procedimento(prontuario: str, procedimento: str) -> dict:
    return {
        "instruction": "Identifique o procedimento realizado.",
        "input": prontuario,
        "output": procedimento,
    }

def criar_tarefa_resumo(prontuario: str, especialidade: str, diagnostico: str, desfecho: str) -> dict:
    resumo = f"Atendimento em {especialidade.lower()}. Diagnóstico: {diagnostico}. Desfecho: {desfecho}."
    return {
        "instruction": "Resuma o atendimento descrito no prontuário.",
        "input": prontuario,
        "output": resumo,
    }

def criar_tarefa_classificacao_multilabel(prontuario: str, especialidade: str, diagnostico: str) -> dict:
    return {
        "instruction": "Classifique a especialidade e identifique o diagnóstico principal do paciente.",
        "input": prontuario,
        "output": {
            "especialidade": especialidade,
            "diagnostico_principal": diagnostico,
        },
    }

dataset_treinamento = []

for i, (paciente, pront) in enumerate(zip(pacientes_sinteticos, prontuarios_sinteticos)):
    texto = pront["prontuario_texto"]
    especialidade = paciente["especialidade"]
    diagnostico = paciente["diagnostico_principal"]
    procedimento = paciente["procedimento_principal"]
    desfecho = paciente["desfecho"]

    # Tarefa 1 — Classificação
    dataset_treinamento.append(criar_tarefa_classificacao(texto, especialidade))

    # Tarefa 2 — Extração de diagnóstico
    dataset_treinamento.append(criar_tarefa_extracao_diagnostico(texto, diagnostico))

    # Tarefa 3 — Extração de procedimento
    dataset_treinamento.append(criar_tarefa_extracao_procedimento(texto, procedimento))

    # Tarefa 4 — Resumo
    dataset_treinamento.append(criar_tarefa_resumo(texto, especialidade, diagnostico, desfecho))

    # Tarefa 5 — Classificação multilabel
    dataset_treinamento.append(criar_tarefa_classificacao_multilabel(texto, especialidade, diagnostico))

print(f"Total de exemplos no dataset: {len(dataset_treinamento)}")

In [ ]:
# ------------------------------------------------------------------------------
# 3.5 Persistência e divisão do dataset via DuckDB
# ------------------------------------------------------------------------------
# Mantemos a geração dos exemplos em Python, mas o dataset oficial é armazenado
# e particionado na mesma conexão DuckDB.
df_dataset_tmp = pd.DataFrame({
    "instruction": [item["instruction"] for item in dataset_treinamento],
    "input": [item["input"] for item in dataset_treinamento],
    "output": [
        json.dumps(item["output"], ensure_ascii=False)
        if isinstance(item["output"], (dict, list))
        else str(item["output"])
        for item in dataset_treinamento
    ],
})

con.register("dataset_treinamento_tmp", df_dataset_tmp)

con.sql("""
CREATE OR REPLACE TABLE dataset_treinamento AS
SELECT
    ROW_NUMBER() OVER () AS id_exemplo,
    instruction,
    input,
    output
FROM dataset_treinamento_tmp
""")

con.unregister("dataset_treinamento_tmp")
del df_dataset_tmp

# Sorteio e divisão feitos pelo DuckDB, com uma coluna de ordem explícita
# (evita depender do pseudo-campo 'rowid', que não é uma garantia estável de
# ordenação entre versões do DuckDB).
con.sql(f"""
CREATE OR REPLACE TABLE dataset_treinamento_particionado AS
WITH embaralhado AS (
    SELECT
        *,
        ROW_NUMBER() OVER (ORDER BY random()) AS ordem,
        COUNT(*) OVER () AS total
    FROM dataset_treinamento
)
SELECT
    id_exemplo,
    instruction,
    input,
    output,
    ordem,
    CASE
        WHEN ordem <= total * {DIVISAO_DATASET["train"]} THEN 'train'
        WHEN ordem <= total * ({DIVISAO_DATASET["train"]} + {DIVISAO_DATASET["validation"]}) THEN 'validation'
        ELSE 'test'
    END AS split
FROM embaralhado
""")

# Recupera cada partição do DuckDB como DataFrame, ordenando de forma
# explícita e estável pela coluna 'ordem' criada acima.
df_train = con.sql("""
SELECT instruction, input, output
FROM dataset_treinamento_particionado
WHERE split = 'train'
ORDER BY ordem
""").df()

df_val = con.sql("""
SELECT instruction, input, output
FROM dataset_treinamento_particionado
WHERE split = 'validation'
ORDER BY ordem
""").df()

df_test = con.sql("""
SELECT instruction, input, output
FROM dataset_treinamento_particionado
WHERE split = 'test'
ORDER BY ordem
""").df()

In [ ]:
def dataframe_para_jsonl(df: pd.DataFrame) -> list[dict]:
    registros = []
    for _, row in df.iterrows():
        output = row["output"]
        try:
            output = json.loads(output)
        except (json.JSONDecodeError, TypeError):
            pass

        registros.append({
            "instruction": row["instruction"],
            "input": row["input"],
            "output": output,
        })
    return registros

train_data = dataframe_para_jsonl(df_train)
val_data = dataframe_para_jsonl(df_val)
test_data = dataframe_para_jsonl(df_test)

# Dataset completo preservando a mesma estrutura dos exemplos originais
dataset_treinamento = train_data + val_data + test_data

n_total = len(dataset_treinamento)

print(f"  Treinamento: {len(train_data)} exemplos ({len(train_data)/n_total*100:.1f}%)")
print(f"  Validação: {len(val_data)} exemplos ({len(val_data)/n_total*100:.1f}%)")
print(f"  Teste: {len(test_data)} exemplos ({len(test_data)/n_total*100:.1f}%)")

# Exporta datasets
if GERAR_JSONL:
    with open(CAMINHO_TRAIN, "w", encoding="utf-8") as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    with open(CAMINHO_VALIDATION, "w", encoding="utf-8") as f:
        for item in val_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    with open(CAMINHO_TEST, "w", encoding="utf-8") as f:
        for item in test_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    # Dataset completo
    with open(CAMINHO_DATASET_COMPLETO, "w", encoding="utf-8") as f:
        for item in dataset_treinamento:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"\n[9] Train/Validation/Test separados e exportados em {DIRETORIO_DATASETS.as_posix()}/")

In [ ]:
# ==============================================================================
# VALIDAÇÃO ESTATÍSTICA
# ==============================================================================
print("\n" + "=" * 80)
print("VALIDAÇÃO ESTATÍSTICA")
print("=" * 80)

print("\n[10] Comparando distribuições...")

# Distribuição sintética por especialidade — calculada no DuckDB
distrib_sintetica_esp = con.sql("""
SELECT
    especialidade,
    COUNT(*) AS quantidade_sintetica,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS percentual_sintetico
FROM pacientes_sinteticos
GROUP BY especialidade
ORDER BY quantidade_sintetica DESC
""").df()

# Merge com distribuição real
comparacao_esp = distrib_especialidade.merge(
    distrib_sintetica_esp,
    on="especialidade",
    how="outer"
)
comparacao_esp["diferenca_percentual"] = (
    comparacao_esp["percentual"] - comparacao_esp["percentual_sintetico"]
).abs()

print("\nComparação por especialidade:")
print(comparacao_esp.to_string(index=False))

# Distribuição sintética por ano — calculada no DuckDB
distrib_sintetica_ano = con.sql("""
SELECT
    ano,
    COUNT(*) AS quantidade_sintetica,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS percentual_sintetico
FROM pacientes_sinteticos
GROUP BY ano
ORDER BY ano
""").df()

comparacao_ano = distrib_ano.merge(
    distrib_sintetica_ano,
    on="ano",
    how="outer"
)
comparacao_ano["diferenca_percentual"] = (
    comparacao_ano["percentual"] - comparacao_ano["percentual_sintetico"]
).abs()

print("\nComparação por ano:")
print(comparacao_ano.to_string(index=False))

# Salva relatório
if GERAR_RELATORIO:
    with open(CAMINHO_RELATORIO_VALIDACAO, "w", encoding="utf-8") as f:
        f.write("=== COMPARAÇÃO POR ESPECIALIDADE ===\n")
        comparacao_esp.to_csv(f, index=False)
        f.write("\n=== COMPARAÇÃO POR ANO ===\n")
        comparacao_ano.to_csv(f, index=False)

    print(f"\nRelatório de validação exportado para: {CAMINHO_RELATORIO_VALIDACAO}")

print("\n[10] Validação estatística concluída.")

In [ ]:
# ==============================================================================
# RESUMO FINAL
# ==============================================================================
print("\n" + "=" * 80)
print("RESUMO FINAL")
print("=" * 80)

print(f"""
[1] ✓ Dados SIH carregados
[2] ✓ Casos de referência encontrados
[3] ✓ Distribuições estatísticas calculadas
[4] ✓ Perfil clínico sintético criado
[5] ✓ Pacientes sintéticos gerados
[6] ✓ Regras de coerência aplicadas
[7] ✓ Prontuários sintéticos gerados
[8] ✓ Dataset de treinamento criado
[9] ✓ Train/Validation/Test separados
[10] ✓ Validação estatística concluída

Arquivos gerados:
  {DIRETORIO_DADOS.as_posix()}/
    ├── casos.parquet
    ├── perfil_*.parquet
    ├── pacientes_sinteticos.parquet
    ├── pacientes_sinteticos.jsonl
    ├── prontuarios_sinteticos.parquet
    ├── prontuarios_sinteticos.jsonl
    └── relatorio_validacao_sintetico.csv

  {DIRETORIO_DATASETS.as_posix()}/
    ├── train.jsonl
    ├── validation.jsonl
    ├── test.jsonl
    └── dataset_treinamento.jsonl

  Banco DuckDB: {ARQUIVO_DUCKDB}
""")

print("=" * 80)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 80)

# Verificação final do banco DuckDB
print("\nTabelas persistidas no DuckDB:")
print(con.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
""").df().to_string(index=False))

# Fecha a conexão de ESCRITA usada por todo o pipeline. A partir daqui, o
# banco fica livre para ser aberto em modo somente-leitura pela ferramenta
# de consulta abaixo — DuckDB não permite uma conexão de escrita e outra de
# leitura abertas ao mesmo tempo sobre o mesmo arquivo.
con.close()
print(f"\nConexão de escrita fechada. Banco disponível em: {ARQUIVO_DUCKDB}")

In [ ]:
# ==============================================================================
# FERRAMENTA DE CONSULTA — FerramentaConsultaSIH
# ==============================================================================
#
# Classe Python pensada para ser usada como "tool" de consulta: por um
# notebook interativo, por outro script, ou por um agente/LLM que precise
# fazer perguntas sobre os dados gerados sem reexecutar o pipeline inteiro.
#
# Abre o banco em modo SOMENTE LEITURA (read_only=True), então pode
# coexistir com múltiplas outras conexões de leitura, e nunca corrompe os
# dados gerados. Todas as consultas livres (`consultar`) são restritas a
# comandos SELECT, para evitar que um uso indevido (ex.: por um agente)
# apague ou corrompa a base.
#
# EXEMPLO DE USO:
#
#   ferramenta = FerramentaConsultaSIH()
#   ferramenta.tabelas()
#   ferramenta.estatisticas_especialidade()
#   ferramenta.amostra_prontuarios(especialidade="OBSTETRICIA", n=2)
#   ferramenta.buscar_paciente("PAC00000001")
#   ferramenta.consultar("SELECT especialidade, COUNT(*) FROM pacientes_sinteticos GROUP BY 1")
#   ferramenta.fechar()
#
# Também pode ser usada como context manager:
#
#   with FerramentaConsultaSIH() as ferramenta:
#       print(ferramenta.resumo_dataset())
#
# ==============================================================================

class FerramentaConsultaSIH:
    """Ferramenta de consulta somente-leitura ao banco DuckDB gerado pelo pipeline."""

    def __init__(self, caminho_banco: str | Path = ARQUIVO_DUCKDB):
        self.caminho_banco = Path(caminho_banco)
        if not self.caminho_banco.exists():
            raise FileNotFoundError(
                f"Banco DuckDB não encontrado em '{self.caminho_banco}'. "
                "Execute o pipeline completo antes de usar a ferramenta de consulta."
            )
        self._con = duckdb.connect(str(self.caminho_banco), read_only=True)

    # --- suporte a "with FerramentaConsultaSIH() as ferramenta:" ---
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.fechar()

    def fechar(self):
        """Fecha a conexão somente-leitura."""
        self._con.close()

    def tabelas(self) -> list[str]:
        """Lista as tabelas e views disponíveis no banco."""
        df = self._con.sql("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
            ORDER BY table_name
        """).df()
        return df["table_name"].tolist()

    def descrever(self, tabela: str) -> pd.DataFrame:
        """Mostra colunas e tipos de uma tabela."""
        self._validar_nome_tabela(tabela)
        return self._con.sql(f"DESCRIBE {tabela}").df()

    def consultar(self, sql: str, limite: int = 500) -> pd.DataFrame:
        """
        Executa uma consulta SQL livre, mas SOMENTE leitura (SELECT ou CTE
        iniciada por WITH). Qualquer outro comando é rejeitado — pensado
        para uso seguro por um agente automatizado.
        """
        sql_normalizado = sql.strip().rstrip(";")
        primeira_palavra = sql_normalizado.split(None, 1)[0].upper() if sql_normalizado else ""

        if primeira_palavra not in ("SELECT", "WITH"):
            raise ValueError(
                "Somente consultas SELECT (ou WITH ... SELECT) são permitidas "
                "nesta ferramenta de consulta somente-leitura."
            )

        # Aplica um limite de segurança caso a consulta não tenha um.
        if "LIMIT" not in sql_normalizado.upper():
            sql_normalizado = f"SELECT * FROM ({sql_normalizado}) AS _sub LIMIT {limite}"

        return self._con.sql(sql_normalizado).df()

    def estatisticas_especialidade(self) -> pd.DataFrame:
        """Distribuição de pacientes sintéticos por especialidade."""
        return self._con.sql("""
            SELECT
                especialidade,
                COUNT(*) AS quantidade,
                ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS percentual
            FROM pacientes_sinteticos
            GROUP BY especialidade
            ORDER BY quantidade DESC
        """).df()

    def estatisticas_diagnostico(self, especialidade: str | None = None, top: int = 10) -> pd.DataFrame:
        """Diagnósticos mais frequentes, opcionalmente filtrados por especialidade."""
        filtro = ""
        if especialidade is not None:
            especialidade_escapada = especialidade.replace("'", "''")
            filtro = f"WHERE especialidade = '{especialidade_escapada}'"

        return self._con.sql(f"""
            SELECT especialidade, diagnostico_principal, COUNT(*) AS quantidade
            FROM pacientes_sinteticos
            {filtro}
            GROUP BY especialidade, diagnostico_principal
            ORDER BY quantidade DESC
            LIMIT {top}
        """).df()

    def amostra_prontuarios(self, especialidade: str | None = None, n: int = 3) -> pd.DataFrame:
        """Retorna N prontuários de exemplo, opcionalmente filtrados por especialidade."""
        filtro = ""
        if especialidade is not None:
            especialidade_escapada = especialidade.replace("'", "''")
            filtro = f"WHERE especialidade = '{especialidade_escapada}'"

        return self._con.sql(f"""
            SELECT id_prontuario, id_paciente, especialidade, prontuario_texto
            FROM prontuarios_sinteticos
            {filtro}
            USING SAMPLE {n}
        """).df()

    def buscar_paciente(self, id_paciente: str) -> dict:
        """Retorna o perfil clínico e o prontuário de um paciente específico."""
        id_escapado = id_paciente.replace("'", "''")

        paciente_df = self._con.sql(f"""
            SELECT * FROM pacientes_sinteticos WHERE id_paciente = '{id_escapado}'
        """).df()

        prontuario_df = self._con.sql(f"""
            SELECT * FROM prontuarios_sinteticos WHERE id_paciente = '{id_escapado}'
        """).df()

        if paciente_df.empty:
            raise ValueError(f"Paciente '{id_paciente}' não encontrado.")

        return {
            "paciente": paciente_df.iloc[0].to_dict(),
            "prontuario": prontuario_df.iloc[0].to_dict() if not prontuario_df.empty else None,
        }

    def resumo_dataset(self) -> dict:
        """Resumo rápido do dataset de treinamento (contagens por split)."""
        try:
            df = self._con.sql("""
                SELECT split, COUNT(*) AS quantidade
                FROM dataset_treinamento_particionado
                GROUP BY split
                ORDER BY split
            """).df()
            return dict(zip(df["split"], df["quantidade"]))
        except duckdb.CatalogException:
            return {"aviso": "Tabela 'dataset_treinamento_particionado' não encontrada no banco."}

    @staticmethod
    def _validar_nome_tabela(tabela: str):
        """Validação simples para evitar injeção de SQL em nomes de tabela."""
        if not tabela.replace("_", "").isalnum():
            raise ValueError(f"Nome de tabela inválido: '{tabela}'")


# Demonstração rápida da ferramenta ao final do pipeline.
print("\n" + "=" * 80)
print("DEMONSTRAÇÃO DA FERRAMENTA DE CONSULTA (FerramentaConsultaSIH)")
print("=" * 80)

with FerramentaConsultaSIH() as ferramenta:
    print("\nTabelas disponíveis:")
    print(ferramenta.tabelas())

    print("\nDistribuição por especialidade (dados sintéticos):")
    print(ferramenta.estatisticas_especialidade().to_string(index=False))

    print("\nResumo do dataset de treinamento (por split):")
    print(ferramenta.resumo_dataset())

print("\nFerramenta 'FerramentaConsultaSIH' pronta para uso em outras células do Colab.")
print("Exemplo: ferramenta = FerramentaConsultaSIH(); ferramenta.buscar_paciente('PAC00000001')")